# Week 9 · Day 2 — Facial Emotion Classification (FER2013) in PyTorch

**Yesterday** you built a digit classifier in **TensorFlow/Keras**. **Today** we do a harder, more interesting problem in **PyTorch** — our main framework from here on — on a *real dataset stored in folders*: recognising **facial emotions**.

The dataset, **FER2013**, is organised the way real image datasets usually are — already split into `train` and `test`, each with one folder per emotion:
```
FER2013/
  train/
    angry/  disgust/  fear/  happy/  neutral/  sad/  surprise/
  test/
    angry/  disgust/  fear/  happy/  neutral/  sad/  surprise/
```
Seven emotions, 48×48 grayscale faces. We’ll load them from disk with **`os`**, then teach a neural network to classify them.

### A note on difficulty (important)
FER2013 is **hard**. Human accuracy on it is only about 65%, and a plain ANN like ours typically reaches **~35–45%**. That sounds low — but random guessing on 7 classes is ~14%, so the network is genuinely learning. The goal today isn’t a high score; it’s to run a real, messy image project end-to-end in PyTorch and understand the tools. **CNNs next week will do much better.**

### Today’s plan
1. **Load images from the folders with `os`.**
2. Prepare the data (the train/test split is already done for us).
3. **Learn PyTorch gently** — tensors, then *you* build the model and training loop.
4. Train and evaluate.
5. **Deep dive: activation functions.**
6. **Deep dive: optimizers.**

---
## 1. Load the images from folders with `os`

The skill: **`os.listdir`** discovers folder contents (class names, then image files), and **`os.path.join`** builds correct file paths on any OS. We never hardcode a filename.

This dataset is **already split**, so we write one loader function and call it twice — once for `train`, once for `test`. That’s cleaner than copy-pasting the loop.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ---- the one path you may need to fix ----
DATA_DIR = "FER2013"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

# class names come from the train subfolders (sorted for a fixed, repeatable order)
class_names = sorted([
    name for name in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, name))
])
print("emotions found:", class_names)
print("number of classes:", len(class_names))

> If the list is empty or wrong, fix `DATA_DIR` so it points at the folder that contains `train` and `test`.

### A loader function we can reuse for both splits
FER2013 images are **48×48 grayscale**. We open each as grayscale (`"L"`), keep it 48×48, and store the pixels with the class index. Writing this once as a function keeps the code clean and is good practice.

In [ ]:
IMG_SIZE = 48   # FER2013 images are 48x48
VALID_EXT = (".jpg", ".jpeg", ".png")

def load_split(split_dir):
    """Walk split_dir/<class>/*.jpg and return (images, labels) arrays."""
    X, y = [], []
    for label_index, class_name in enumerate(class_names):
        class_folder = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_folder):
            continue
        loaded = 0
        for filename in os.listdir(class_folder):
            if not filename.lower().endswith(VALID_EXT):
                continue
            img_path = os.path.join(class_folder, filename)
            try:
                img = Image.open(img_path).convert("L")        # grayscale
                img = img.resize((IMG_SIZE, IMG_SIZE))          # ensure 48x48
                X.append(np.array(img))
                y.append(label_index)
                loaded += 1
            except Exception as e:
                print(f"  skipped {filename}: {e}")
        print(f"  {class_name:10s}: {loaded}")
    return np.array(X, dtype="float32"), np.array(y)

In [ ]:
# this reads ~35k images total, so it may take a minute — that's normal
print("loading train:")
X_train_img, y_train = load_split(TRAIN_DIR)
print("loading test:")
X_test_img, y_test = load_split(TEST_DIR)

print("\ntrain images:", X_train_img.shape)   # (~28709, 48, 48)
print("test images: ", X_test_img.shape)    # (~7178, 48, 48)

### Look at the data first (always)

In [ ]:
plt.figure(figsize=(12, 4))
for label_index, class_name in enumerate(class_names):
    idx = np.where(y_train == label_index)[0][0]   # first image of this emotion
    plt.subplot(1, len(class_names), label_index + 1)
    plt.imshow(X_train_img[idx], cmap="gray")
    plt.title(class_name, fontsize=9)
    plt.axis("off")
plt.suptitle("One face per emotion")
plt.tight_layout()
plt.show()

# class balance — FER2013 is imbalanced (few 'disgust', many 'happy')
print("training images per emotion:")
for label_index, class_name in enumerate(class_names):
    print(f"  {class_name:10s}: {(y_train == label_index).sum()}")

Notice the **class imbalance** — FER2013 has thousands of `happy` faces but only a few hundred `disgust`. That’s realistic (real data is rarely balanced) and it’s one reason the task is hard: the network sees far fewer examples of the rare emotions.

---
## 2. Prepare the data

The train/test split is **already done** (that’s what the folders gave us), so unlike the previous notebooks we skip `train_test_split`. We just:
1. **Flatten** each `48×48` image to a `2304`-vector.
2. **Scale** pixels 0–255 → 0–1 (divide by 255).

In [ ]:
# 1. flatten: (N, 48, 48) -> (N, 2304)
X_train = X_train_img.reshape(X_train_img.shape[0], -1)
X_test  = X_test_img.reshape(X_test_img.shape[0], -1)

# 2. scale 0-255 -> 0-1
X_train = X_train / 255.0
X_test  = X_test / 255.0

n_features = X_train.shape[1]   # 2304
n_classes = len(class_names)    # 7
print("train:", X_train.shape, "  test:", X_test.shape)
print("inputs per image:", n_features, "  classes:", n_classes)

**The two numbers your model must match:** input = **`n_features`** (2304), output = **`n_classes`** (7).

---
## 3. PyTorch, gently

### 3a. Tensors
A **tensor** is PyTorch’s trainable NumPy array. Images become float tensors; labels become `long` integers.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print("X_train_t:", X_train_t.shape, X_train_t.dtype)
print("first label:", y_train_t[0].item(), "->", class_names[y_train_t[0].item()])

> **Labels stay integers** (`0`–6), not one-hot. PyTorch’s `CrossEntropyLoss` does the one-hot step internally — simpler than Keras’s `to_categorical`.

### 3b. Batches with `DataLoader`
This dataset is large, so batching really matters here. `DataLoader` serves shuffled batches of 64.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

print("training images:", len(train_ds))
print("batches per epoch:", len(train_loader))

### 3c. Build your network  ✍️ (Task A)

**You** write the model — you’ve seen the pattern all week: an `nn.Sequential` of `nn.Linear` layers with `nn.ReLU()` between them.

**Requirements:**
- First layer input = **`n_features`**.
- At least **one hidden layer** with **`nn.ReLU()`** after it. For a harder dataset like this, a couple of decent-sized hidden layers help.
- Last layer output = **`n_classes`**. **No softmax** — `CrossEntropyLoss` handles it.

**Template:**
```python
model = nn.Sequential(
    nn.Linear(n_features, 512),   # input -> hidden
    nn.ReLU(),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, n_classes)     # -> 7, no activation
)
```

💡 *Each layer’s output size must equal the next layer’s input size.*

In [ ]:
torch.manual_seed(42)

# ===== YOUR CODE HERE (Task A) =====
# Build your network and name it exactly `model`.

model = None   # <-- replace with your nn.Sequential(...)

# ===================================

print(model)

### Self-check (provided ✅)

In [ ]:
assert model is not None, "Task A: you haven't built `model` yet."
with torch.no_grad():
    out = model(torch.randn(5, n_features))
assert out.shape == (5, n_classes), f"Output {tuple(out.shape)} should be (5, {n_classes}). Check layer sizes."
print(f"✅ shape check passed: {n_features} in -> {n_classes} out. Ready to train.")

### 3d. Loss and optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("loss and optimizer ready")

### 3e. The training loop  ✍️ (read carefully)

The **four moves**, once per batch: **forward → loss → backward → update**. We provide this as a reusable function (you’ll call it again in the experiments) — read every line.

In [ ]:
def train_model(model, optimizer, loss_fn, loader,
                X_test_t, y_test_t, epochs=20, log=True):
    """Train a model; return per-epoch train loss and test accuracy."""
    train_losses, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            preds = model(xb)              # 1. forward
            loss = loss_fn(preds, yb)      # 2. loss
            optimizer.zero_grad()          # 3. backward...
            loss.backward()
            optimizer.step()               # 4. update
            running += loss.item()
        train_losses.append(running / len(loader))
        model.eval()
        with torch.no_grad():
            acc = (model(X_test_t).argmax(1) == y_test_t).float().mean().item()
        test_accs.append(acc)
        if log:
            print(f"epoch {epoch+1:2d}  train loss {train_losses[-1]:.4f}  test acc {acc:.2%}")
    return train_losses, test_accs

---
## 4. Train and evaluate

This trains on ~28k images — give it a moment per epoch. Expect test accuracy to climb into the **~35–45%** range and then plateau. That plateau is the ceiling of a plain ANN on faces; it’s the honest result, and the motivation for CNNs.

In [ ]:
train_losses, test_accs = train_model(
    model, optimizer, loss_fn, train_loader, X_test_t, y_test_t, epochs=20)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, color="purple", marker="o")
ax1.set_title("Training loss"); ax1.set_xlabel("epoch"); ax1.grid(alpha=0.3)
ax2.plot(test_accs, color="green", marker="o")
ax2.set_title("Test accuracy"); ax2.set_xlabel("epoch"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Watch for a telling shape: **train loss keeps falling while test accuracy flattens** — that’s the network starting to **overfit** (memorising training faces without generalising). Very common on FER2013 with a plain ANN.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).argmax(dim=1)
test_acc = (test_pred == y_test_t).float().mean().item()
print(f"TEST ACCURACY: {test_acc:.2%}  (random guess on 7 classes = ~14%)")

cm = confusion_matrix(y_test_t, test_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"Confusion matrix — {test_acc:.1%}")
plt.tight_layout()
plt.show()

The confusion matrix is worth reading closely: the network usually does best on **happy** and **surprise** (visually distinctive) and struggles with **fear/sad/neutral**, which look alike even to people. It also rarely predicts **disgust** — too few training examples. These are exactly the mistakes a human makes too.

In [ ]:
# look at some predictions — green = correct, red = wrong
n_show = 10
plt.figure(figsize=(13, 5))
for i in range(n_show):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_test_img[i], cmap="gray")
    t, p = y_test_t[i].item(), test_pred[i].item()
    plt.title(f"{class_names[p]}\n(true: {class_names[t]})", color="green" if t == p else "red", fontsize=8)
    plt.axis("off")
plt.suptitle("Predictions")
plt.tight_layout()
plt.show()

You built a full facial-emotion classifier in PyTorch — loading raw images from folders with `os`, all the way to evaluation. Now the two ideas that shape how well it trains.

---
## 5. Deep dive: Activation functions

An **activation function** is the non-linear step inside each neuron (every `nn.ReLU()`).

### Why we need them
Without an activation, stacked layers collapse into a single straight line — useless for a task like faces. The activation adds the **bend** that lets the network model complex patterns (the Week 8 XOR reason).

The three you’ll meet most:

In [ ]:
z = torch.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, out, desc) in zip(axes, [
    ("ReLU",    torch.relu(z),    "0 for negatives, linear for positives"),
    ("Sigmoid", torch.sigmoid(z), "squashes into (0, 1)"),
    ("Tanh",    torch.tanh(z),    "squashes into (-1, 1)")]):
    ax.plot(z, out, color="purple", linewidth=2)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name); ax.set_xlabel(desc, fontsize=9); ax.grid(alpha=0.3)
plt.suptitle("The three activation functions you'll use most")
plt.tight_layout()
plt.show()

| Activation | Output range | Good for | Watch out for |
|---|---|---|---|
| **ReLU** | 0 to ∞ | **hidden layers** (modern default) | “dying” neurons stuck at 0 |
| **Sigmoid** | 0 to 1 | a **binary** output | saturates — slows learning |
| **Tanh** | −1 to 1 | hidden layers (older nets, RNNs) | also saturates at extremes |

**Rule:** ReLU in hidden layers by default; sigmoid on a binary output; no final activation for multi-class (let `CrossEntropyLoss` handle it).

### See it matter
Same network, three hidden activations, compared. *(We use 8 epochs here to keep the comparison quick on this large dataset.)*

In [ ]:
def make_model(activation):
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 512), activation(),
        nn.Linear(512, 128),        activation(),
        nn.Linear(128, n_classes))

results_act = {}
for name, act in [("ReLU", nn.ReLU), ("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh)]:
    m = make_model(act)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    _, accs = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=8, log=False)
    results_act[name] = accs
    print(f"{name:8s} final test acc: {accs[-1]:.2%}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, accs in results_act.items():
    plt.plot(range(1, len(accs)+1), accs, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("test accuracy")
plt.title("Activation functions compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Typically **ReLU and tanh learn faster** than **sigmoid**, whose gradients shrink at large inputs (saturation), slowing early learning — the reason ReLU is the hidden-layer default. *(Exact numbers vary per run; the pattern is the lesson.)*

---
## 6. Deep dive: Optimizers

The **optimizer** updates the weights after gradients are computed. Same goal, different stepping:
- **SGD:** straight downhill; simple, can be slow.
- **SGD + Momentum:** keeps speed from prior steps; smoother, faster.
- **Adam:** adapts the step per weight; usually the fastest starter and safest default.

Race them on the same network:

In [ ]:
def make_relu_model():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 512), nn.ReLU(),
        nn.Linear(512, 128),        nn.ReLU(),
        nn.Linear(128, n_classes))

optimizers = {
    "SGD":          lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD+Momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam":         lambda p: torch.optim.Adam(p, lr=0.001),
}

results_opt = {}
for name, make_opt in optimizers.items():
    m = make_relu_model()
    opt = make_opt(m.parameters())
    losses, _ = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=8, log=False)
    results_opt[name] = losses
    print(f"{name:14s} final train loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, losses in results_opt.items():
    plt.plot(range(1, len(losses)+1), losses, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("Optimizers compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Usually **plain SGD** falls slowest, **Momentum** speeds it up, and **Adam** drops fastest early — why Adam is the common default. (A well-tuned SGD+Momentum sometimes generalises better; knowing the trade-off is the skill.)

### The shared knob: learning rate
The step size — the most important number to get roughly right. Too big overshoots; too small crawls.

In [ ]:
for lr in [0.0001, 0.001, 0.01, 0.1]:
    m = make_relu_model()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses, accs = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=5, log=False)
    print(f"lr={lr:<7}  final train loss {losses[-1]:.4f}   test acc {accs[-1]:.2%}")

Usually the middle learning rates (0.001–0.01) win; the smallest underfits in a few epochs and the largest is unstable. Same lesson as Week 8 — now across optimizers.

---
## Your turn (practice) ✍️

Make changes and observe. Pick at least two:

1. **Add `nn.Dropout(0.3)`** between hidden layers in Task A to fight the overfitting you saw. Does the train/test gap shrink?
2. **Try a bigger first hidden layer** (e.g. 1024). Does test accuracy improve, or just overfit faster?
3. **Try `nn.LeakyReLU()`** vs `nn.ReLU()` on the comparison chart.
4. **Train the main model for 40 epochs** — where does test accuracy plateau?

Write a sentence under each about what you saw.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

**Loading a pre-split dataset from folders with `os`:**
- `os.listdir` → class names and image files; `os.path.join` → correct paths on any OS.
- The `train`/`test` folders **are** the split — no `train_test_split` needed. One reusable `load_split` function, called twice.
- Images resized to 48×48 grayscale, flattened to 2304, scaled by /255.

**PyTorch:** tensors (labels stay integer with `CrossEntropyLoss`), `nn.Sequential`/`nn.Linear`/`nn.ReLU` (no final softmax), and a training loop you write — forward → loss → backward → update.

**Activation functions:** ReLU (hidden default), sigmoid (binary output), tanh (older) — ReLU learns fastest.

**Optimizers:** SGD → +Momentum → Adam (best default), sharing the critical **learning rate** knob.

**The honest result:** a plain ANN reaches only ~35–45% on FER2013, and overfits, because flattening throws away the 2D structure of a face. **Next week’s CNNs keep that structure and do far better** — this notebook is exactly the baseline that shows why they’re needed.